Workflow for Spartacus data, Works similar to INCA data

In [4]:
from dotenv import load_dotenv
import os
from hera.workflows import models, CronWorkflow, script, Artifact, Parameter, DAG, Steps, Step, NoneArchiveStrategy, Workflow
from hera.shared import global_config

load_dotenv("/home/simon/work/zarr_workflows/.env")

True

In [5]:
global_config.host = "https://services.eodc.eu/workflows/"
global_config.namespace = "spartacus"
global_config.token = os.getenv("argo_token_prod")
global_config.image = "ghcr.io/oscipal/image_zarr:latest"

In [6]:
nfs_volume = [models.Volume(
    name="eodc-mount",
    persistent_volume_claim={"claimName": "eodc-nfs-claim"},
    )]

security_context = {"runAsUser": 74268,
                    "runAsGroup": 71473}

In [7]:
@script(volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")])

def prepare_zarr(store_path: str = "/eodc/products/eodc/geosphere_spartacus/SPARTACUS.zarr"):
    import datetime
    import numpy as np
    import zarr
 
    now = datetime.datetime.now()
    now_np = np.datetime64(now).astype('datetime64[D]')
    origin = np.datetime64("1961-01-01").astype("datetime64[D]")
 
    new_shape = int((now_np-origin).astype(int))
    new_extent = np.arange(0,new_shape,1)
 
    store = zarr.storage.LocalStore(store_path)
    group = zarr.group(store=store)
 
    array_names=set(group.array_keys())
    coords = {"time", "x", "y"}
    data_arrays = array_names-coords
 
    group["time"].resize(new_shape)
    for array in data_arrays:
        group_shape  = group[array].shape
        group[array].resize((new_shape, group_shape[1], group_shape[2]))
 
    zarr.consolidate_metadata(store)
    store = zarr.storage.LocalStore(store_path)
    group = zarr.group(store=store)
 
    group["time"][:]=new_extent


In [8]:
@script(outputs=Artifact(name="spartacus-file", path="/tmp/output.nc", archive=NoneArchiveStrategy()))

def spartacus_download(variable: str):
    from urllib.request import urlretrieve
    import datetime

    today = datetime.date.today() - datetime.timedelta(months=10)
    year = today.strftime('%Y')
    
    url = f"https://public.hub.geosphere.at/datahub/resources/spartacus-v2-1d-1km/filelisting/{variable}/SPARTACUS2-DAILY_{variable}_{year}.nc"
    urlretrieve(url, "/tmp/output.nc")

In [9]:
@script(
    inputs=Artifact(name="spartacus-file", path="/tmp/output.nc"),
    volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")]
)

def spartacus_write(variable: str):
    import xarray as xr
    import numpy as np
    import zarr
    import datetime

    fill_value = -999
    artifact_path = "/tmp/output.nc" 

    today = datetime.date.today() - datetime.timedelta(days=3)
    year = today.strftime('%Y')

    ds_nc = xr.open_dataset(artifact_path, mask_and_scale=False).load()
    zarr_path = "/eodc/products/eodc/geosphere_spartacus/SPARTACUS.zarr"

    store = zarr.storage.LocalStore(zarr_path)
    group = zarr.group(store=store)
    zarr.consolidate_metadata(store)

    data = ds_nc[variable].values

    ref_date = np.datetime64("1961-01-01")
    start_date = np.datetime64(f"{year}-01-01")
    start_idx = int((start_date - ref_date) / np.timedelta64(1, "D"))
    num_days = data.shape[0]
    end_idx = start_idx + num_days

    group[variable][start_idx:end_idx, :, :] = data

In [10]:
items = ["TX", "TN", "RR", "SA"]

# At first a CronWorkflow is created with the necessary inputs.
with CronWorkflow(
    generate_name="spartacus-yearly-",
    schedule="0 6 1 6 *",
    volumes = nfs_volume,
    security_context=security_context,
    entrypoint="workflow"
) as w:
    
    # Secondly, a DAG is defined which shall be executed for each parameter. The inputs are defined in the Steps below. So this DAG acts like a function being defined and executed in a different step.
    with DAG(name="pipeline", inputs=[Parameter(name="args")]) as pipeline:
        
        # The arguments for the scripts are passed as a dictionary, wherease the arguments for the 'param' parameter are taken from the input of the DAG
        download = spartacus_download(arguments={"variable":"{{inputs.parameters.args}}"},)

        # The Artifact written with download also has to be given to the function.
        process = spartacus_write(arguments=[{"variable": "{{inputs.parameters.args}}"}, 
                                        download.get_artifact("spartacus-file").with_name("spartacus-file")],)

        # Here the sequence of the steps is defined
        download >> process

    # As we defined "workflow" as the entrypoint in the CronWorkflow this part gets executed first, in contrast to DAGs Steps will be executed in the order they are in
    with Steps(name="workflow"):
        # First the time dimension is extended in the zarr store
        prepare_zarr()

        # Now the DAG is executed, it is used as a template, passing the inca_parameters as with_param and using "{{item}}" in arguments the DAG will be executed parallel for each parameter.
        Step(name="parallel-pipelines", template=pipeline, with_param=items, arguments={"args":"{{item}}"})

In [11]:
with open("spartacus_workflow.yaml", "w") as f:
    f.write(w.to_yaml())

In [12]:
w.create()

CronWorkflow(api_version=None, kind=None, metadata=ObjectMeta(annotations=None, creation_timestamp=Time(__root__=datetime.datetime(2026, 1, 30, 10, 52, 28, tzinfo=datetime.timezone.utc)), deletion_grace_period_seconds=None, deletion_timestamp=None, finalizers=None, generate_name='spartacus-yearly-', generation=1, labels={'workflows.argoproj.io/creator': 'system-serviceaccount-default-jenkins'}, managed_fields=[ManagedFieldsEntry(api_version='argoproj.io/v1alpha1', fields_type='FieldsV1', fields_v1=FieldsV1(), manager='argo', operation='Update', subresource=None, time=Time(__root__=datetime.datetime(2026, 1, 30, 10, 52, 28, tzinfo=datetime.timezone.utc)))], name='spartacus-yearly-4858m', namespace='spartacus', owner_references=None, resource_version='154915181', self_link=None, uid='8207bd02-d5dc-4a98-83f0-6344319397ec'), spec=CronWorkflowSpec(concurrency_policy=None, failed_jobs_history_limit=None, schedule='0 6 1 6 *', schedules=None, starting_deadline_seconds=None, stop_strategy=None